This notebook keep all Activity nodes and relations but sets every Activity input to zero in training, validation and all test patterns.

In [1]:
import sys
import os
import torch
import pickle 
import numpy as np
import pandas as pd
import json
from os.path import dirname 

In [2]:
root_path = os.getcwd()
sys.path.insert(0, root_path + "/data")
print(root_path)

/home/danbi/Projects/SANAGRAPH


In [3]:
from pipeline import train_hgnn, test_hgnn

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [13]:
dataset = "bpi_2012_CZ"
num_runs = 10
num_epochs = 100

nan_methods = ["odd", "even", "random", "window", "attr_level"]
data_dir = root_path + "/data/"
data_dir_graphs = data_dir + f"datasets/comuzzi/graphs_repair/{dataset}/"
hyp_opt_dir = root_path + "/results_CAISE/hyp_opt/"
results_root = root_path + f"/results_CAISE/activity-zero/{dataset}/"
save_path = results_root + "ablation_Activity_zero/"
os.makedirs(save_path, exist_ok=True)

print(data_dir)
print(data_dir_graphs)
print(hyp_opt_dir)
print(results_root)
print(save_path)

/home/danbi/Projects/SANAGRAPH/data/
/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/bpi_2012_CZ/
/home/danbi/Projects/SANAGRAPH/results_CAISE/hyp_opt/
/home/danbi/Projects/SANAGRAPH/results_CAISE/activity-zero/bpi_2012_CZ/
/home/danbi/Projects/SANAGRAPH/results_CAISE/activity-zero/bpi_2012_CZ/ablation_Activity_zero/


In [14]:
with open(data_dir + "dataset_features.json") as f:
    dataset_info = json.load(f)[dataset]

categorical_columns = [k for k in dataset_info["categorical"] if k != "Activity"]
real_value_columns = [k for k in dataset_info["numerical"] if k != "Activity"]

In [15]:
configs = pd.read_csv(hyp_opt_dir + f"{dataset}_CONFIGS.csv")
best_row = configs.sort_values(by="AVG_total_loss").iloc[0]
best_parameters = {
    "hid": int(best_row["hid"]),
    "layers": int(best_row["layers"]),
    "lr": float(best_row["lr"]),
    "batch_size": int(best_row["batch_size"]),
    "weight_decay": float(best_row["weight_decay"]),
    "aggregation": best_row["aggregation"],
}
print(best_parameters)

{'hid': 128, 'layers': 4, 'lr': 0.0001, 'batch_size': 128, 'weight_decay': 0.01, 'aggregation': 'sum'}


In [16]:

from torch_geometric.transforms import ToUndirected

### Training and validation graphs

Keeping Activity nodes and relationships for messagee passing, removing Activity from labels and masks used for predictions

In [17]:
with open(data_dir_graphs + f"{dataset}_TRAIN_V2_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + f"{dataset}_VALID_V2_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/home/danbi/Projects/SANAGRAPH/data/datasets/comuzzi/graphs_repair/bpi_2012_CZ/bpi_2012_CZ_TRAIN_V2_repair.pkl'

In [18]:
from tqdm.auto import tqdm

In [ ]:
transform = ToUndirected()

with torch.no_grad():
    for graphs in [X_train, X_valid]:
        for i, g in enumerate(graphs):
            assert "Activity" in g.node_types, "Use full-model graphs."
            g["Activity"].x = torch.zeros_like(g["Activity"].x)
            g.y = {k: v for k, v in g.y.items() if k != "Activity"}
            g.masks = {k: v for k, v in g.masks.items() if k != "Activity"}
            graphs[i] = transform(g)

edge_types = sorted({
    edge for graphs in [X_train, X_valid]
    for g in graphs for edge in g.edge_types
})
outputcat = {k: X_train[0][k].x.shape[1] for k in categorical_columns}
outputreal = real_value_columns
print("Prediction targets:", categorical_columns + real_value_columns)

### Training and test

In [ ]:
results = {pattern: [] for pattern in nan_methods}

for run in tqdm(range(num_runs)):
    random.seed(run)
    np.random.seed(run)
    torch.manual_seed(run)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(run)

    net = train_hgnn(
        best_parameters, outputcat, outputreal,
        edge_types=edge_types, X_train=X_train, X_valid=X_valid,
        device=device, epochs=num_epochs,
    )

    for pattern in nan_methods:
        with open(data_dir_graphs + f"{dataset}_TEST_V2_repair_{pattern}.pkl", "rb") as f:
            X_test = pickle.load(f)

        with torch.no_grad():
            for i, g in enumerate(X_test):
                assert "Activity" in g.node_types, "Use full-model graphs."
                g["Activity"].x = torch.zeros_like(g["Activity"].x)
                g.y = {k: v for k, v in g.y.items() if k != "Activity"}
                g.masks = {k: v for k, v in g.masks.items() if k != "Activity"}
                X_test[i] = transform(g)

        results[pattern].append(test_hgnn(
            net, outputcat, outputreal, device=device, test_graphs=X_test
        ))
        del X_test

    summary_rows = []
    for pattern in nan_methods:
        table = pd.DataFrame(results[pattern])
        means = table.mean()
        stds = table.std()
        table.to_csv(save_path + f"{pattern}_V2_RESULTS_END.csv", index=False)
        pd.DataFrame({"mean": means, "std": stds}).to_csv(
            save_path + f"{pattern}_V2_MEAN_STD_END.csv"
        )
        summary_rows.append({
            "variant": "Activity_zero", "excluded_feature": "Activity",
            "test_type": pattern, "n_runs": len(table), **means.to_dict()
        })
    pd.DataFrame(summary_rows).to_csv(results_root + "ablation_summary.csv", index=False)
    print(f"Saved run {run + 1}/{num_runs}")
    del net